In [1]:
from pathlib import Path
import re
import pandas as pd

RY_TO_EV = 13.605693009

energy_pattern = re.compile(
    r"^\s*!?\s*total energy\s*=\s*([-+]?\d*\.?\d+(?:[eEdD][-+]?\d+)?)\s*Ry",
    re.MULTILINE,
)


def last_total_energy_ev(path: Path) -> float:
    text = path.read_text(errors="ignore")
    matches = energy_pattern.findall(text)
    if not matches:
        raise ValueError(f"No total energy found in {path}")
    return float(matches[-1].replace("D", "E").replace("d", "e")) * RY_TO_EV


# 1) Slab reference energies
slab_files = {
    "TiN": Path("../slab/output_files/TiN_slab.out"),
    "VN": Path("../slab/output_files/VN_slab.out"),
}
slab_energy_ev = {surface: last_total_energy_ev(path) for surface, path in slab_files.items()}

# 2) Gas/isolated adsorbate reference energies
adsorbates = ["Li2S", "Li2S2", "Li2S4", "Li2S6", "Li2S8", "S8"]
ads_energy_ev = {
    ads: last_total_energy_ev(Path("../adsorbates") / ads / "espresso.pwo")
    for ads in adsorbates
}

# 3) Combined slab+adsorption energies (pick highest u file as final)
outputs_dir = Path("outputs")
combi_pattern = re.compile(
    r"^(?P<surface>TiN|VN)_(?P<ads>Li2S|Li2S2|Li2S4|Li2S6|Li2S8|S8)_combi_u(?P<u>\d+)\.out$"
)

final_out_by_system = {}
for out_file in outputs_dir.glob("*_combi_u*.out"):
    m = combi_pattern.match(out_file.name)
    if not m:
        continue

    key = (m.group("surface"), m.group("ads"))
    u_value = int(m.group("u"))

    if key not in final_out_by_system or u_value > final_out_by_system[key][0]:
        final_out_by_system[key] = (u_value, out_file)

rows = []
for (surface, ads), (u_value, out_file) in sorted(final_out_by_system.items()):
    e_combi = last_total_energy_ev(out_file)
    e_ads = e_combi - slab_energy_ev[surface] - ads_energy_ev[ads]

    rows.append(
        {
            "surface": surface,
            "adsorbate": ads,
            "u_final": u_value,
            "combi_out": out_file.name,
            "E_combi_eV": e_combi,
            "E_slab_eV": slab_energy_ev[surface],
            "E_adsorbate_eV": ads_energy_ev[ads],
            "E_adsorption_eV": e_ads,
        }
    )

if not rows:
    raise FileNotFoundError(f"No matching combined output files found in {outputs_dir}")

results = pd.DataFrame(rows)

# Optional convenient sorting
ads_order = {name: i for i, name in enumerate(adsorbates)}
results = results.sort_values(
    by=["surface", "adsorbate"],
    key=lambda c: c.map(ads_order) if c.name == "adsorbate" else c,
).reset_index(drop=True)

# Save a csv copy
csv_path = Path("final_xyz") / "adsorption_energies.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(csv_path, index=False)

print("Slab reference energies (eV):")
for k, v in slab_energy_ev.items():
    print(f"  {k}: {v:.8f}")

print("\nAdsorbate reference energies (eV):")
for k in adsorbates:
    print(f"  {k}: {ads_energy_ev[k]:.8f}")

print("\nAdsorption energies (eV):")
display(results[["surface", "adsorbate", "u_final", "E_adsorption_eV"]])

print(f"\nSaved full table to: {csv_path}")

Slab reference energies (eV):
  TiN: -136471.78410307
  VN: -161796.60994840

Adsorbate reference energies (eV):
  Li2S: -721.74969707
  Li2S2: -1050.51554766
  Li2S4: -1707.41513869
  Li2S6: -2363.40238650
  Li2S8: -3019.11626628
  S8: -2622.86915815

Adsorption energies (eV):


,surface,adsorbate,u_final,E_adsorption_eV
0,TiN,Li2S,1,-0.759363
1,TiN,Li2S2,1,-2.732890
2,TiN,Li2S4,3,-2.325538
3,TiN,Li2S6,5,-3.612426
4,TiN,Li2S8,4,-4.603765
5,TiN,S8,3,-7.046553
6,VN,Li2S,1,-8.813646
7,VN,Li2S2,1,-9.315206
8,VN,Li2S4,1,-8.911151
9,VN,Li2S6,7,-13.099987



Saved full table to: final_xyz/adsorption_energies.csv
